In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
token = os.getenv("HF_TOKEN")
login(token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
import torch
import torch.nn as nn
from transformers import AutoModel
from src.re_make_Gate import inject_gating
from src.model import dinosplus_classfier

# ── 모델 로드
backbone = AutoModel.from_pretrained("facebook/dinov3-vits16plus-pretrain-lvd1689m")

# ── gate 주입 (전체 레이어)
backbone, hooks = inject_gating(
    backbone,
    gate_type="elementwise",
    keep_cls_ungated=True,
    target_layers=None
)

# ── 분류기 감싸기
vit = dinosplus_classfier(backbone, num=200)

# ── 전체 파라미터 학습
for p in vit.parameters():
    p.requires_grad = True

print("모델 준비 완료!")
print(f"학습 파라미터 수: {sum(p.numel() for p in vit.parameters() if p.requires_grad):,}")

Loading weights:   0%|          | 0/235 [00:00<?, ?it/s]

[inject_gating] 총 12개 중 12개 레이어에 G1 gate 적용
  → 적용 인덱스: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]  (gate_type=elementwise)
모델 준비 완료!
학습 파라미터 수: 30,766,664


In [4]:
from datasets import load_dataset
from src.data import setdata
from torch.utils.data import DataLoader

data = load_dataset('zh-plus/tiny-imagenet')
train = setdata(data['train'])
test  = setdata(data['valid'])

train_loader = DataLoader(train, batch_size=128, pin_memory=True, num_workers=4, shuffle=True)
test_loader  = DataLoader(test,  batch_size=128, pin_memory=True, num_workers=4, shuffle=False)

In [5]:
import torch.optim as optim

optimizer = optim.Adam(vit.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
scaler    = torch.amp.GradScaler()

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

device: cuda


In [7]:
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(False)

In [ ]:
from src.training import train

train(
    vit,
    train_loader,
    test_loader,
    50,
    criterion,
    scaler,
    device,
    optimizer,
    'full finetuning dinov3 splus gated gate_loss',
    '/home/hyuksu/deep-learning-study/outputs/best_dinov3_splus_full_gate full finetuning.pth'
)

2026/04/01 21:51:57 INFO mlflow.tracking.fluent: Experiment with name 'full finetuning dinov3 splus gated gate_loss' does not exist. Creating a new experiment.


  0%|          | 0/50 [00:00<?, ?it/s]